In [1]:
import os
import sys
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.linear_model import ElasticNetCV, LassoCV, LinearRegression, RidgeCV
from torch.utils.data import DataLoader, TensorDataset

# sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from data_preprocessing import RANDOM_STATE, prepare_data
from evaluation import evaluate_all, metrics, nn_predict, save, to_dollar
from mlp import MLP

from train import train_nn, train_baselines, train_neural_networks

In [2]:
data = prepare_data()
print(f"\nDataset: {data['n_samples']} samples × {data['n_features']} features")
print(f"Split: train={data['n_train']}, val={data['n_val']}, test={data['n_test']}")


Dataset: 2927 samples × 15 features
Split: train=1872, val=469, test=586


In [3]:
print("\n" + "=" * 70)
print("  Step 1: Baseline (OLS / Ridge / Lasso / ElasticNet)")
print("=" * 70)
linears, bl_results = train_baselines(data)


  Step 1: Baseline (OLS / Ridge / Lasso / ElasticNet)
  OLS           RMSE=$    28,265  MAE=$    17,917  NRMSE%= 15.44  MAPE%=  9.54  R²=0.9009
  Ridge         RMSE=$    28,292  MAE=$    17,922  NRMSE%= 15.46  MAPE%=  9.54  R²=0.9007  alpha=13.53
  Lasso         RMSE=$    28,385  MAE=$    17,910  NRMSE%= 15.51  MAPE%=  9.53  R²=0.9000  alpha=0.002442
  ElasticNet    RMSE=$    28,385  MAE=$    17,910  NRMSE%= 15.51  MAPE%=  9.53  R²=0.9000  alpha=0.002442 l1_ratio=1.00

  Best baseline: OLS  R²=0.9009


In [4]:
from evaluation import plot_diagnostics

for name in ["Ridge", "Lasso"]:
    pred = bl_results[name]["y_pred"]
    plot_diagnostics(data["y_test"], pred, f"Baseline: {name}  R²={bl_results[name]['R2']:.4f}", f"step5_{name.lower()}_diagnostics.png")
    print(f"Saved {name} diagnostic plot")

Saved Ridge diagnostic plot
Saved Lasso diagnostic plot


In [5]:
print("\n" + "=" * 70)
print("  Step 3: Neural Network Experiments")
print("=" * 70)
nn_models, nn_hists, nn_configs = train_neural_networks(data)

evaluate_all(data, nn_models, nn_hists, nn_configs, linears, bl_results)



  Step 3: Neural Network Experiments

  Training A: Shallow [64] ...
    epochs=62 (best=32)  time=0.5s  RMSE=$22,524  NRMSE%=12.31  R²=0.9371

  Training B: Medium [128-64] ...
    epochs=59 (best=29)  time=0.5s  RMSE=$21,777  NRMSE%=11.90  R²=0.9412

  Training C: Deep [256-128-64] ...
    epochs=51 (best=21)  time=0.6s  RMSE=$21,977  NRMSE%=12.01  R²=0.9401

  Training D: Deep+Sigmoid ...
    epochs=65 (best=35)  time=0.8s  RMSE=$25,728  NRMSE%=14.06  R²=0.9179

  Training E: Deep+Dropout0.3 ...
    epochs=64 (best=34)  time=1.1s  RMSE=$20,934  NRMSE%=11.44  R²=0.9456

  Training F: Deep+L2 ...
    epochs=45 (best=15)  time=0.7s  RMSE=$22,602  NRMSE%=12.35  R²=0.9366

  Step 4: Training Diagnostics
  Saved architecture comparison curves
  Saved activation comparison
  Saved regularization comparison

  Step 5: All Models Comparison
               Model         RMSE          MAE    NRMSE%    MAPE%       R2   Type
  E: Deep+Dropout0.3 20933.988089 14475.537716 11.438304 8.291444 0.94